# Stage 6 — SNOMED CT Grounding

**Input** : `stage_05_evidence_scoring/scored_branches.json` per admission  
**Output**: `stage_06_snomed_grounding/grounded_symptoms.json` per admission

## What this stage does
Maps high-confidence symptoms (`route_to_snomed=True`) to ICD-10-CM codes
via SNOMED CT ontology — replacing Stage 5's keyword matching with semantic grounding.

## SNOMED CT API options (all free)

| Option | API | Registration | URL |
|--------|-----|--------------|-----|
| `'auto'` | tries A→B→C in order | — | |
| `'bioportal'` | NCBO BioPortal | https://bioportal.bioontology.org/accounts/new | |
| `'umls'` | NLM UMLS REST | https://uts.nlm.nih.gov/uts/signup-login | |
| `'snowstorm'` | IHTSDO Snowstorm | No key (but often blocked by firewalls) | |

**→ Start with `SNOMED_API_OPTION = 'auto'` and paste whichever keys you have. The notebook picks the first working one.**

## Pipeline position
```
Stage 5: Evidence Scoring + Pruning   ✅
Stage 6: SNOMED Grounding             ← THIS NOTEBOOK
Stage 6b: History Context Builder     (next)
Stage 7:  Final ICD Decision          (next)
```

## 1. Setup — API key loading

Keys are loaded automatically from three sources (in order):
1. **`.env` file** (recommended) — create `ai-agents-for-clinical-coding/.env` with your key, never commit it
2. **Environment variable** — set `BIOPORTAL_API_KEY` in your OS/terminal
3. **Runtime prompt** — if neither above is set, you'll be asked to type it once (not saved)

You never need to paste a key into this notebook.

In [1]:
import json
import os
import time
import urllib.parse
import requests
from pathlib import Path
from collections import defaultdict
import pandas as pd

# ── API selection ──────────────────────────────────────────────────────────────
# 'auto' tries snowstorm → bioportal → umls and picks the first that works
SNOMED_API_OPTION = 'umls' #'auto'

# ── Key loading ────────────────────────────────────────────────────────────────
# Priority: .env file → OS environment variable → runtime prompt
# Never paste keys directly into this notebook.
#
# .env file location: ai-agents-for-clinical-coding/.env
# Format (one per line, no quotes):
#   BIOPORTAL_API_KEY=your-key-here
#   UMLS_API_KEY=your-key-here

PROJECT_ROOT = Path(r'C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding')
ENV_FILE     = PROJECT_ROOT / '.env'

def _load_env_file(path):
    env = {}
    if path.exists():
        for line in path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                k, _, v = line.partition('=')
                env[k.strip()] = v.strip()
    return env

def _get_key(name, env_dict):
    return env_dict.get(name) or os.environ.get(name, '')

_env              = _load_env_file(ENV_FILE)
BIOPORTAL_API_KEY = _get_key('BIOPORTAL_API_KEY', _env)
UMLS_API_KEY      = _get_key('UMLS_API_KEY', _env)

# Report source (never print the key value itself)
for _name, _val in [('BIOPORTAL_API_KEY', BIOPORTAL_API_KEY), ('UMLS_API_KEY', UMLS_API_KEY)]:
    if _val:
        _src = '.env file' if _env.get(_name) else 'environment variable'
        print(f'  {_name}: loaded from {_src}')
    else:
        print(f'  {_name}: not set')

if not BIOPORTAL_API_KEY and not UMLS_API_KEY:
    print()
    print('No keys found. Options:')
    print(f'  a) Create {ENV_FILE}')
    print('     Add line: BIOPORTAL_API_KEY=your-key-here')
    print('     Then re-run this cell.')
    print()
    print('  b) Enter key now (not saved anywhere):')
    _inp = input('  BioPortal API key (Enter to skip): ').strip()
    if _inp:
        BIOPORTAL_API_KEY = _inp
        print('  Key accepted for this session only.')

# ── Paths ──────────────────────────────────────────────────────────────────────
RECORDS_DIR   = PROJECT_ROOT / 'patient_records'
STAGE5_OUTPUT = 'stage_05_evidence_scoring'
STAGE6_OUTPUT = 'stage_06_snomed_grounding'
CACHE_FILE    = RECORDS_DIR / 'snomed_cache.json'

# ── Load cache ─────────────────────────────────────────────────────────────────
if CACHE_FILE.exists():
    with open(CACHE_FILE) as f:
        SNOMED_CACHE = json.load(f)
    print(f'Cache loaded: {len(SNOMED_CACHE)} cached terms')
else:
    SNOMED_CACHE = {}
    print('No cache — will build during this run')

patients = sorted([p for p in RECORDS_DIR.iterdir() if p.is_dir() and p.name.startswith('patient_')])
print(f'Patients found: {len(patients)}')

  BIOPORTAL_API_KEY: loaded from .env file
  UMLS_API_KEY: loaded from .env file
Cache loaded: 243 cached terms
Patients found: 15


## 2. API Functions

In [2]:
# ── Shared ────────────────────────────────────────────────────────────────────
API_DELAY   = 0.2
TIMEOUT     = 10
MAX_RETRIES = 1
JSON_HEADER = {'Accept': 'application/json'}

ICD10CM_REFSET_ID = '6011000124106'


def _get(url, params=None, headers=None, timeout=None):
    h = headers or JSON_HEADER
    t = timeout or TIMEOUT
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, params=params, headers=h, timeout=t)
            if r.status_code == 200:
                return r.json()
            if r.status_code == 429:
                time.sleep(5 * (attempt + 1))
        except Exception:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(1)
    return None


# ══ Option A: IHTSDO Snowstorm ════════════════════════════════════════════════

SNOWSTORM_BASE   = 'https://snowstorm.ihtsdotools.org/snowstorm/snomed-ct'
SNOWSTORM_BRANCH = 'MAIN'


def snowstorm_search(term, n=3):
    ck = f'snow_c:{term.lower()}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{SNOWSTORM_BASE}/{SNOWSTORM_BRANCH}/concepts',
                {'term': term, 'activeFilter': 'true', 'limit': n}, timeout=6)
    time.sleep(API_DELAY)
    results = []
    if data and 'items' in data:
        for item in data['items']:
            pt = (item.get('pt') or item.get('fsn') or {}).get('term', '')
            results.append({'conceptId': item.get('conceptId', ''), 'pt': pt})
    SNOMED_CACHE[ck] = results
    return results


def snowstorm_icd_map(concept_id):
    ck = f'snow_icd:{concept_id}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{SNOWSTORM_BASE}/{SNOWSTORM_BRANCH}/members',
                {'referenceSet': ICD10CM_REFSET_ID,
                 'referencedComponentId': concept_id,
                 'active': 'true', 'limit': 20})
    time.sleep(API_DELAY)
    results = []
    if data and 'items' in data:
        for item in data['items']:
            af = item.get('additionalFields', {})
            code = af.get('mapTarget', '')
            if code:
                results.append({'icdCode': code.replace('.', ''),
                                'mapPriority': int(af.get('mapPriority', 1)),
                                'mapAdvice': af.get('mapAdvice', ''),
                                'icd_source': 'snomed_direct_map'})
        results.sort(key=lambda x: x['mapPriority'])
    SNOMED_CACHE[ck] = results
    return results


def snowstorm_parents(concept_id):
    ck = f'snow_par:{concept_id}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{SNOWSTORM_BASE}/browser/{SNOWSTORM_BRANCH}/concepts/{concept_id}/parents',
                {'form': 'inferred'})
    time.sleep(API_DELAY)
    results = []
    if data and isinstance(data, list):
        for item in data[:3]:
            pt  = (item.get('pt') or {}).get('term', '')
            cid = item.get('conceptId', '')
            if cid:
                results.append({'conceptId': cid, 'pt': pt})
    SNOMED_CACHE[ck] = results
    return results


# ══ Option B: NCBO BioPortal ══════════════════════════════════════════════════

BP_BASE = 'https://data.bioontology.org'


def bioportal_search(term, n=3):
    if not BIOPORTAL_API_KEY:
        return []
    ck = f'bp_c:{term.lower()}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{BP_BASE}/search',
                {'q': term, 'ontologies': 'SNOMEDCT', 'apikey': BIOPORTAL_API_KEY,
                 'pagesize': n, 'exact_match': 'false'})
    time.sleep(API_DELAY)
    results = []
    if data and 'collection' in data:
        for item in data['collection'][:n]:
            uri = item.get('@id', '')
            cid = uri.split('/')[-1] if uri else ''
            results.append({'conceptId': cid, 'pt': item.get('prefLabel', ''), 'uri': uri})
    SNOMED_CACHE[ck] = results
    return results


def bioportal_icd_map(concept_uri):
    if not BIOPORTAL_API_KEY or not concept_uri:
        return []
    ck = f'bp_icd:{concept_uri}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    encoded = urllib.parse.quote(concept_uri, safe='')
    data = _get(f'{BP_BASE}/ontologies/SNOMEDCT/classes/{encoded}/mappings',
                {'apikey': BIOPORTAL_API_KEY})
    time.sleep(API_DELAY)
    results = []
    if data and isinstance(data, list):
        for i, mapping in enumerate(data):
            for cls in mapping.get('classes', []):
                cls_id = cls.get('@id', '')
                if 'ICD10CM' in cls_id or 'ICD-10-CM' in cls_id:
                    code = cls_id.split('/')[-1].replace('.', '')
                    if code:
                        results.append({'icdCode': code.upper(),
                                        'mapPriority': i + 1,
                                        'mapAdvice': '',
                                        'icd_source': 'snomed_direct_map'})
    SNOMED_CACHE[ck] = results
    return results


# ══ Option C: NLM UMLS REST API ═══════════════════════════════════════════════

UMLS_BASE    = 'https://uts-ws.nlm.nih.gov/rest'
UMLS_VERSION = 'current'


def umls_search(term, n=3):
    if not UMLS_API_KEY:
        return []
    ck = f'umls_c:{term.lower()}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{UMLS_BASE}/search/{UMLS_VERSION}',
                {'string': term, 'sabs': 'SNOMEDCT_US', 'returnIdType': 'concept',
                 'apiKey': UMLS_API_KEY, 'pageSize': n})
    time.sleep(API_DELAY)
    results = []
    if data and 'result' in data:
        for item in data['result'].get('results', [])[:n]:
            results.append({'conceptId': item.get('ui', ''), 'pt': item.get('name', '')})
    SNOMED_CACHE[ck] = results
    return results


def umls_icd_map(cui):
    # Get ICD10CM atoms for a UMLS CUI — symptom-level codes.
    if not UMLS_API_KEY or not cui:
        return []
    ck = f'umls_icd2:{cui}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{UMLS_BASE}/content/{UMLS_VERSION}/CUI/{cui}/atoms',
                {'sabs': 'ICD10CM', 'apiKey': UMLS_API_KEY, 'pageSize': 10})
    time.sleep(API_DELAY)
    results = []
    if data and 'result' in data:
        seen = set()
        for i, item in enumerate(data['result'][:10]):
            code_url = item.get('code', '')
            code = code_url.split('/')[-1] if '/' in code_url else code_url
            code = code.replace('.', '').upper()
            if code and code not in seen:
                seen.add(code)
                results.append({'icdCode': code, 'mapPriority': i + 1,
                                'mapAdvice': item.get('name', ''),
                                'icd_source': 'snomed_direct_map'})
    SNOMED_CACHE[ck] = results
    return results


def umls_search_icd_direct(term, n=5):
    # Search ICD10CM directly by term string to capture diagnosis-level codes.
    # Complements umls_icd_map which returns symptom-level codes via SNOMED.
    # e.g. "pneumonia" → J189, J159 (diagnoses) even if SNOMED crosswalk misses them.
    if not UMLS_API_KEY:
        return []
    ck = f'umls_icd_direct:{term.lower()}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{UMLS_BASE}/search/{UMLS_VERSION}',
                {'string': term, 'sabs': 'ICD10CM', 'returnIdType': 'code',
                 'apiKey': UMLS_API_KEY, 'pageSize': n})
    time.sleep(API_DELAY)
    results = []
    if data and 'result' in data:
        seen = set()
        for i, item in enumerate(data['result'].get('results', [])[:n]):
            code = item.get('ui', '').replace('.', '').upper()
            if code and code not in seen:
                seen.add(code)
                results.append({'icdCode': code, 'mapPriority': i + 1,
                                'mapAdvice': item.get('name', ''),
                                'icd_source': 'icd_direct_search'})
    SNOMED_CACHE[ck] = results
    return results


def umls_get_associated_icds(cui, n=3):
    """Get ICD codes for conditions associated with a CUI via UMLS RO relations.
    Example: 'liver failure' CUI → hepatic encephalopathy → K7682."""
    if not UMLS_API_KEY or not cui:
        return []
    ck = f'umls_assoc:{cui}'
    if ck in SNOMED_CACHE:
        return SNOMED_CACHE[ck]
    data = _get(f'{UMLS_BASE}/content/{UMLS_VERSION}/CUI/{cui}/relations',
                {'apiKey': UMLS_API_KEY, 'pageSize': 20})
    time.sleep(API_DELAY)
    results = []
    if data and 'result' in data:
        seen = set()
        for rel in data['result'][:20]:
            if rel.get('relationLabel') != 'RO':
                continue
            related_url = rel.get('relatedId', '')
            related_cui = related_url.split('/')[-1] if '/' in related_url else related_url
            if not related_cui.startswith('C'):
                continue
            icd_codes = umls_icd_map(related_cui)
            for c in icd_codes[:2]:
                if c['icdCode'] not in seen:
                    seen.add(c['icdCode'])
                    results.append({**c,
                                    'icd_source': 'associated_finding',
                                    'mapAdvice': rel.get('relatedIdName', '')})
            if len(results) >= n:
                break
    SNOMED_CACHE[ck] = results
    return results


# ══ Active API ════════════════════════════════════════════════════════════════
ACTIVE_API = None


def search_concept(term):
    if ACTIVE_API == 'snowstorm': return snowstorm_search(term)
    if ACTIVE_API == 'bioportal': return bioportal_search(term)
    if ACTIVE_API == 'umls':      return umls_search(term)
    return []


def get_icd_map(concept):
    if ACTIVE_API == 'snowstorm': return snowstorm_icd_map(concept['conceptId'])
    if ACTIVE_API == 'bioportal': return bioportal_icd_map(concept.get('uri', ''))
    if ACTIVE_API == 'umls':      return umls_icd_map(concept['conceptId'])
    return []


def get_parents(concept_id):
    if ACTIVE_API == 'snowstorm': return snowstorm_parents(concept_id)
    return []


def save_cache():
    with open(CACHE_FILE, 'w') as f:
        json.dump(SNOMED_CACHE, f, indent=2)


print('API functions defined.')

API functions defined.


## 3. Auto-detect working SNOMED API

Run this cell first — it tries each option in order and picks the first that works.

If **all fail**, you need a free API key:
- **BioPortal** (fastest): https://bioportal.bioontology.org/accounts/new → key emailed instantly  
- **UMLS** (~5 min): https://uts.nlm.nih.gov/uts/signup-login  

Paste whichever key you get into cell 1, then re-run this cell.

In [3]:
API_AVAILABLE = False
ACTIVE_API    = None
TEST_TERM     = 'chest pain'

def _test_api(name, search_fn, icd_fn, concept_key='conceptId'):
    global ACTIVE_API, API_AVAILABLE
    print(f'  Testing {name} ...', end=' ', flush=True)
    try:
        concepts = search_fn(TEST_TERM)
        if not concepts:
            print('✗ no concepts returned')
            return False
        top  = concepts[0]
        icds = icd_fn(top)
        if icds:
            print(f'✓  concept="{top["pt"]}"  ICD={icds[0]["icdCode"]}')
        else:
            print(f'✓  concept="{top["pt"]}"  (no ICD map — hierarchy will be tried)')
        ACTIVE_API    = name
        API_AVAILABLE = True
        return True
    except Exception as e:
        print(f'✗ {type(e).__name__}: {str(e)[:80]}')
        return False


order = (
    [SNOMED_API_OPTION]
    if SNOMED_API_OPTION != 'auto'
    else ['snowstorm', 'bioportal', 'umls']
)

print(f'Auto-detecting SNOMED CT API (order: {order})')
print('=' * 60)

for api in order:
    if api == 'snowstorm':
        if _test_api('snowstorm',
                     lambda t: snowstorm_search(t),
                     lambda c: snowstorm_icd_map(c['conceptId'])):
            break

    elif api == 'bioportal':
        if not BIOPORTAL_API_KEY:
            print('  Testing bioportal ... ✗ no API key (register at https://bioportal.bioontology.org/accounts/new)')
        elif _test_api('bioportal',
                       lambda t: bioportal_search(t),
                       lambda c: bioportal_icd_map(c.get('uri', ''))):
            break

    elif api == 'umls':
        if not UMLS_API_KEY:
            print('  Testing umls       ... ✗ no API key (register at https://uts.nlm.nih.gov/uts/signup-login)')
        elif _test_api('umls',
                       lambda t: umls_search(t),
                       lambda c: umls_icd_map(c['conceptId'])):
            break

print()
if API_AVAILABLE:
    print(f'✓ Active SNOMED API: {ACTIVE_API}')
    print('  Proceed to run grounding (cell 5).')
else:
    print('✗ No SNOMED API available.')
    print()
    print('  ACTION REQUIRED — get a free API key:')
    print('  Option 1 (fastest): BioPortal')
    print('    → Go to: https://bioportal.bioontology.org/accounts/new')
    print('    → Fill in name + email. Key is emailed immediately.')
    print('    → Paste it into BIOPORTAL_API_KEY in cell 1, re-run cell 1 and this cell.')
    print()
    print('  Option 2: NLM UMLS')
    print('    → Go to: https://uts.nlm.nih.gov/uts/signup-login')
    print('    → Register with institutional email. Key shown after email verification.')
    print('    → Paste it into UMLS_API_KEY in cell 1, re-run cell 1 and this cell.')

Auto-detecting SNOMED CT API (order: ['umls'])
  Testing umls ... ✓  concept="Chest Pain"  ICD=R079

✓ Active SNOMED API: umls
  Proceed to run grounding (cell 5).


## 4. Grounding Functions

In [4]:
# Terms too generic to produce useful ICD codes — skip SNOMED entirely for these.
# The underlying diagnosis (e.g. sepsis, pneumonia) should be captured as its own term.
GENERIC_SYMPTOM_TERMS = {
    'fever', 'pain', 'nausea', 'vomiting', 'fatigue', 'weakness',
    'malaise', 'rash', 'swelling', 'cough', 'dizziness', 'headache',
    'chills', 'sweating', 'loss of appetite', 'shortness of breath',
    'weight loss', 'weight gain', 'confusion', 'lethargy', 'edema',
    'nausea and vomiting', 'dyspnea', 'chest pain', 'abdominal pain',
    'back pain', 'joint pain', 'muscle pain', 'sore throat', 'runny nose',
    'increased edema', 'generalized weakness', 'general weakness',
}


def _is_generic(term):
    return term.strip().lower() in GENERIC_SYMPTOM_TERMS


def ground_symptom(symptom, force=False):
    """Ground one symptom to ICD codes.

    force=True bypasses the generic-term filter — used as fallback when every
    term in an admission is generic and we need at least some predictions.
    """
    term = symptom.get('term', '')
    result = {
        'term'             : term,
        'symptom_score'    : symptom.get('score', 0),
        'status'           : symptom.get('status', ''),
        'severity'         : symptom.get('severity', ''),
        'evidence'         : symptom.get('evidence', ''),
        'snomed_concept_id': None,
        'snomed_label'     : None,
        'snomed_ancestors' : [],
        'icd_candidates'   : [],
        'icd_source'       : 'none',
    }
    if not term:
        return result

    # Skip generic symptom terms — too vague for reliable ICD coding
    if not force and _is_generic(term):
        result['icd_source'] = 'skipped_generic_symptom'
        return result

    # ── Path 1: SNOMED concept → ICD crosswalk (symptom-level codes) ──────────
    concepts = search_concept(term)
    if concepts:
        top = concepts[0]
        result['snomed_concept_id'] = top['conceptId']
        result['snomed_label']      = top['pt']

        icd_codes = get_icd_map(top)
        if icd_codes:
            result['icd_candidates'] = icd_codes
            result['icd_source']     = 'snomed_direct_map'
        else:
            parents = get_parents(top['conceptId'])
            result['snomed_ancestors'] = [p['pt'] for p in parents]
            for parent in parents:
                parent_codes = get_icd_map(parent)
                if parent_codes:
                    for c in parent_codes:
                        c['icd_source'] = 'snomed_ancestor_map'
                    result['icd_candidates'] = parent_codes
                    result['icd_source']     = 'snomed_ancestor_map'
                    break
    else:
        result['icd_source'] = 'no_snomed_concept'

    # ── Path 2: direct ICD10CM text search (diagnosis-level codes) ────────────
    if ACTIVE_API == 'umls':
        direct = umls_search_icd_direct(term)
        if direct:
            existing = {c['icdCode'] for c in result['icd_candidates']}
            for c in direct:
                if c['icdCode'] not in existing:
                    result['icd_candidates'].append(c)
                    existing.add(c['icdCode'])

    # ── Path 3: associated findings via UMLS RO relations ─────────────────────
    # e.g. "liver failure" → hepatic encephalopathy → K7682
    if ACTIVE_API == 'umls' and result['snomed_concept_id']:
        assoc = umls_get_associated_icds(result['snomed_concept_id'])
        if assoc:
            existing = {c['icdCode'] for c in result['icd_candidates']}
            for c in assoc:
                if c['icdCode'] not in existing:
                    result['icd_candidates'].append(c)
                    existing.add(c['icdCode'])

    if not result['icd_candidates'] and result['icd_source'] not in ('no_snomed_concept', 'skipped_generic_symptom'):
        result['icd_source'] = 'no_icd_map'
    elif result['icd_candidates'] and result['icd_source'] == 'none':
        result['icd_source'] = 'icd_direct_search'

    return result


def _is_kept(branch):
    v = branch.get('keep', False)
    return v is True or v == 'True'


def ground_admission(scored_path):
    with open(scored_path) as f:
        scored = json.load(f)

    grounded_branches = []
    for branch in scored.get('scored_branches', []):
        if not _is_kept(branch):
            continue

        symptoms = branch.get('symptoms', [])
        has_flag = any('route_to_snomed' in s for s in symptoms)
        if has_flag:
            to_ground = [s for s in symptoms if s.get('route_to_snomed', False)]
        else:
            to_ground = symptoms

        grounded_syms = [ground_symptom(s) for s in to_ground]

        # Fallback: if every term was skipped as generic, force-ground top 2 by score
        total_icd = sum(len(s['icd_candidates']) for s in grounded_syms)
        if total_icd == 0 and to_ground:
            top2 = sorted(to_ground, key=lambda s: s.get('score', 0), reverse=True)[:2]
            grounded_syms = [ground_symptom(s, force=True) for s in top2]

        if grounded_syms:
            grounded_branches.append({
                'category'         : branch['category'],
                'branch_score'     : branch['branch_score'],
                'keep_reason'      : branch.get('keep_reason', ''),
                'grounded_symptoms': grounded_syms,
            })

    n_direct   = sum(1 for b in grounded_branches for s in b['grounded_symptoms'] if 'direct'   in s['icd_source'])
    n_ancestor = sum(1 for b in grounded_branches for s in b['grounded_symptoms'] if 'ancestor' in s['icd_source'])
    n_none     = sum(1 for b in grounded_branches for s in b['grounded_symptoms'] if 'no_'      in s['icd_source'] or s['icd_source'] == 'none')

    return {
        'patient_id'         : scored.get('patient_id'),
        'admission_id'       : scored.get('admission_id'),
        'snomed_api'         : ACTIVE_API,
        'grounded_branches'  : grounded_branches,
        'n_branches'         : len(grounded_branches),
        'n_grounded_symptoms': sum(len(b['grounded_symptoms']) for b in grounded_branches),
        'n_direct_maps'      : n_direct,
        'n_ancestor_maps'    : n_ancestor,
        'n_no_map'           : n_none,
    }


print('Grounding functions defined.')

Grounding functions defined.


## 5. Run Grounding on All 15 Patients

In [5]:
if not API_AVAILABLE:
    raise RuntimeError(
        'No SNOMED API available.\n'
        'Get a free BioPortal key at https://bioportal.bioontology.org/accounts/new\n'
        'or a UMLS key at https://uts.nlm.nih.gov/uts/signup-login\n'
        'Then paste it into cell 1 and re-run cells 1 and 3.'
    )

all_grounded = []

for patient_dir in patients:
    patient_id = patient_dir.name.replace('patient_', '')
    adm_dirs   = sorted((patient_dir / 'admissions').iterdir()) \
                 if (patient_dir / 'admissions').exists() else []

    for adm_dir in adm_dirs:
        scored_path = adm_dir / STAGE5_OUTPUT / 'scored_branches.json'
        if not scored_path.exists():
            print(f'  SKIP {patient_id}/{adm_dir.name} — run Stage 5 first')
            continue

        print(f'Patient {patient_id} | {adm_dir.name} ...', end=' ', flush=True)
        grounded = ground_admission(scored_path)

        out_dir = adm_dir / STAGE6_OUTPUT
        out_dir.mkdir(exist_ok=True)
        with open(out_dir / 'grounded_symptoms.json', 'w') as f:
            json.dump(grounded, f, indent=2, default=str)

        all_grounded.append(grounded)
        print(f'{grounded["n_grounded_symptoms"]} symptoms '
              f'[direct={grounded["n_direct_maps"]} '
              f'ancestor={grounded["n_ancestor_maps"]} '
              f'no_map={grounded["n_no_map"]}]')

save_cache()
print(f'\nDone. {len(all_grounded)} admissions grounded via {ACTIVE_API}.')
print(f'Cache: {len(SNOMED_CACHE)} entries saved.')

Patient 10361982 | hadm_24286431 ... 1 symptoms [direct=0 ancestor=0 no_map=1]
Patient 10426859 | hadm_29908281 ... 5 symptoms [direct=5 ancestor=0 no_map=0]
Patient 10458324 | hadm_21744342 ... 6 symptoms [direct=5 ancestor=0 no_map=1]
Patient 11251337 | hadm_29568708 ... 5 symptoms [direct=3 ancestor=0 no_map=1]
Patient 11474876 | hadm_29672491 ... 1 symptoms [direct=0 ancestor=0 no_map=1]
Patient 11607177 | hadm_23293838 ... 2 symptoms [direct=0 ancestor=0 no_map=2]
Patient 12007928 | hadm_23749816 ... 1 symptoms [direct=1 ancestor=0 no_map=0]
Patient 13196707 | hadm_21475988 ... 4 symptoms [direct=3 ancestor=0 no_map=0]
Patient 13508515 | hadm_21834271 ... 3 symptoms [direct=1 ancestor=0 no_map=2]
Patient 13952483 | hadm_23852410 ... 7 symptoms [direct=6 ancestor=0 no_map=0]
Patient 16014068 | hadm_29042843 ... 2 symptoms [direct=2 ancestor=0 no_map=0]
Patient 17774110 | hadm_27339772 ... 7 symptoms [direct=4 ancestor=0 no_map=3]
Patient 18412100 | hadm_26093939 ... 4 symptoms [dir

## 6. Inspect One Admission

In [6]:
EXAMPLE_IDX = 11
ex = all_grounded[EXAMPLE_IDX]

print(f'Patient   : {ex["patient_id"]}')
print(f'Admission : {ex["admission_id"]}')
print(f'API used  : {ex["snomed_api"]}')
print(f'Branches  : {ex["n_branches"]}  |  Symptoms grounded: {ex["n_grounded_symptoms"]}')
print(f'Maps      : direct={ex["n_direct_maps"]}  ancestor={ex["n_ancestor_maps"]}  no_map={ex["n_no_map"]}')
print()

for branch in ex['grounded_branches']:
    print(f'BRANCH: {branch["category"].upper()}  (score={branch["branch_score"]})')
    for sym in branch['grounded_symptoms']:
        print(f'  Term      : {sym["term"]}  [{sym["status"]}]')
        print(f'  SNOMED    : {sym["snomed_concept_id"]}  —  {sym["snomed_label"]}')
        if sym['snomed_ancestors']:
            print(f'  Ancestors : {sym["snomed_ancestors"]}')
        print(f'  ICD source: {sym["icd_source"]}')
        for c in sym['icd_candidates'][:5]:
            print(f'    {c["icdCode"]:12s}  priority={c["mapPriority"]}  {c.get("mapAdvice","")[:60]}')
        print()

Patient   : 17774110
Admission : 27339772
API used  : umls
Branches  : 7  |  Symptoms grounded: 7
Maps      : direct=4  ancestor=0  no_map=3

BRANCH: INFECTIOUS  (score=2.8)
  Term      : Septic shock  [present]
  SNOMED    : C0036983  —  Septic Shock
  ICD source: icd_direct_search
    R6520         priority=1  Severe sepsis without septic shock
    T8112         priority=2  Postprocedural septic shock
    R6521         priority=3  Severe sepsis with septic shock
    T8112XS       priority=4  Postprocedural septic shock, sequela
    T8112XA       priority=5  Postprocedural septic shock, initial encounter

BRANCH: RESPIRATORY  (score=2.8)
  Term      : Hypoxic respiratory failure  [present]
  SNOMED    : None  —  None
  ICD source: no_snomed_concept

BRANCH: NEUROLOGIC  (score=2.7)
  Term      : Hepatic encephalopathy  [present]
  SNOMED    : C0019151  —  Hepatic Encephalopathy
  ICD source: snomed_direct_map
    K7682         priority=1  Hepatic encephalopathy

BRANCH: RENAL  (score=2

## 7. Evaluate vs Ground Truth

In [7]:
def load_ground_truth(patient_dir, hadm_id):
    gt_path = patient_dir / 'admissions' / f'hadm_{hadm_id}' / 'ground_truth.txt'
    if not gt_path.exists():
        return []
    codes = []
    with open(gt_path, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            # Lines look like: "   1. A419 — Sepsis, unspecified organism"
            if line and line[0].isdigit() and '—' in line:
                # Split on em-dash, take left side; split on dot to drop list number
                code = line.split('—')[0].split('.')[-1].strip().split()[0].upper()
                if code:
                    codes.append(code)
    return codes


def _prefix_match(pred, gt_set):
    """True if pred equals a GT code, or either is a prefix of the other."""
    if pred in gt_set:
        return True
    for gt in gt_set:
        if gt.startswith(pred) or pred.startswith(gt):
            return True
    return False


def evaluate_grounded(grounded, ground_truth):
    predicted = set()
    for branch in grounded['grounded_branches']:
        for sym in branch['grounded_symptoms']:
            for c in sym.get('icd_candidates', []):
                predicted.add(c['icdCode'].replace('.', '').upper())

    gt_set = set(ground_truth)

    # Prefix-aware true positives: I95 counts as hit if GT has I959, and vice versa
    tp = {p for p in predicted if _prefix_match(p, gt_set)}

    precision = len(tp) / len(predicted) if predicted else 0.0
    recall    = len(tp) / len(gt_set)    if gt_set    else 0.0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0.0

    return {
        'predicted'      : sorted(predicted),
        'ground_truth'   : sorted(gt_set),
        'true_positives' : sorted(tp),
        'false_positives': sorted(p for p in predicted if not _prefix_match(p, gt_set)),
        'false_negatives': sorted(gt_set - predicted),
        'precision'      : round(precision, 3),
        'recall'         : round(recall, 3),
        'f1'             : round(f1, 3),
    }


all_metrics_s6 = []

for grounded in all_grounded:
    pid  = grounded['patient_id']
    hadm = grounded['admission_id']
    pdir = RECORDS_DIR / f'patient_{pid}'
    gt   = load_ground_truth(pdir, hadm)
    if not gt:
        print(f'  ! No ground truth: patient={pid} hadm={hadm}')
        continue
    m = evaluate_grounded(grounded, gt)
    m['patient_id']   = pid
    m['admission_id'] = hadm
    m['n_gt_codes']   = len(gt)
    m['n_predicted']  = len(m['predicted'])
    all_metrics_s6.append(m)

    out_dir = pdir / 'admissions' / f'hadm_{hadm}' / STAGE6_OUTPUT
    out_dir.mkdir(exist_ok=True)
    with open(out_dir / 'evaluation.json', 'w') as f:
        json.dump(m, f, indent=2)

if not all_metrics_s6:
    print('No evaluations produced.')
    print(f'  all_grounded has {len(all_grounded)} entries')
    print(f'  n_grounded_symptoms per admission: {[g["n_grounded_symptoms"] for g in all_grounded]}')
else:
    df6 = pd.DataFrame(all_metrics_s6)[['patient_id', 'n_gt_codes', 'n_predicted', 'precision', 'recall', 'f1']]
    print(df6.to_string(index=False))
    print(f'\nMean Precision : {df6["precision"].mean():.3f}')
    print(f'Mean Recall    : {df6["recall"].mean():.3f}')
    print(f'Mean F1        : {df6["f1"].mean():.3f}')
    print(f'\nStage 5 keyword baseline F1 : 0.037')
    print(f'SNOMED improvement          : +{round(df6["f1"].mean() - 0.037, 3)}')

patient_id  n_gt_codes  n_predicted  precision  recall    f1
  10361982           5            0      0.000   0.000 0.000
  10426859          22           10      0.000   0.000 0.000
  10458324          12           30      0.167   0.417 0.238
  11251337           7           12      0.000   0.000 0.000
  11474876          17            0      0.000   0.000 0.000
  11607177          13            0      0.000   0.000 0.000
  12007928          19            5      0.000   0.000 0.000
  13196707          32           11      0.091   0.031 0.047
  13508515          14            5      0.000   0.000 0.000
  13952483          25           20      0.200   0.160 0.178
  16014068          19           11      0.091   0.053 0.067
  17774110          29           15      0.200   0.103 0.136
  18412100           8            5      1.000   0.625 0.769
  19104262          10            0      0.000   0.000 0.000
  19632936          15           10      0.200   0.133 0.160

Mean Precision : 0.130


## 8. Save Summary

In [8]:
if not all_metrics_s6:
    print('No metrics to summarise — run cell 7 first and ensure it produces results.')
else:
    tier_counts = defaultdict(int)
    for g in all_grounded:
        for b in g['grounded_branches']:
            for s in b['grounded_symptoms']:
                tier_counts[s.get('icd_source', 'none')] += 1

    summary = {
        'stage'               : 'stage_06_snomed_grounding',
        'snomed_api'          : ACTIVE_API,
        'n_patients'          : len(patients),
        'n_admissions'        : len(all_metrics_s6),
        'mean_precision'      : round(df6['precision'].mean(), 3),
        'mean_recall'         : round(df6['recall'].mean(), 3),
        'mean_f1'             : round(df6['f1'].mean(), 3),
        'stage5_baseline_f1'  : 0.037,
        'icd_source_breakdown': dict(tier_counts),
        'per_patient'         : all_metrics_s6,
    }

    out_path = RECORDS_DIR / 'stage_06_summary.json'
    with open(out_path, 'w') as f:
        json.dump(summary, f, indent=2, default=str)

    print(f'Saved: {out_path}')
    print(f'\nICD source breakdown:')
    for src, count in sorted(tier_counts.items()):
        print(f'  {src:30s}: {count}')
    print(f'\nStage 6 Mean F1 : {summary["mean_f1"]}')
    print(f'Stage 5 baseline: 0.028')
    print(f'Improvement     : +{round(summary["mean_f1"] - 0.028, 3)}')

Saved: C:\Users\esnam\OneDrive\Desktop\esna_master_proj\ai-agents-for-clinical-coding\patient_records\stage_06_summary.json

ICD source breakdown:
  icd_direct_search             : 6
  no_icd_map                    : 6
  no_snomed_concept             : 9
  skipped_generic_symptom       : 5
  snomed_direct_map             : 27

Stage 6 Mean F1 : 0.106
Stage 5 baseline: 0.028
Improvement     : +0.078
